# Flight 与 GTMD2 统一 UN 区域数据

固定使用 22 个 UN 最细地理区域，构建 Flight + GTMD2 统一国家交集。

### 1. 构建统一国家交集

Flight、GTMD2、IHME、Sequences 四者取交集，确保所有数据源覆盖同一组国家。

In [1]:
from pathlib import Pathimport pandas as pdFLIGHT_OUTPUT_DIR = Path('un_subregion_flight')GTMD_OUTPUT_DIR = Path('un_subregion_gtmd')GTMD_PATH = Path('flight_data/GTMD2_trips.csv')GTMD_YEARS = [2019, 2020]for output_dir in (FLIGHT_OUTPUT_DIR, GTMD_OUTPUT_DIR):    (output_dir / 'processed_data').mkdir(parents=True, exist_ok=True)    (output_dir / 'figures').mkdir(parents=True, exist_ok=True)UN_REGION_ORDER = [    'Australia and New Zealand', 'Caribbean', 'Central America', 'Central Asia',    'Eastern Africa', 'Eastern Asia', 'Eastern Europe', 'Melanesia', 'Micronesia',    'Middle Africa', 'Northern Africa', 'Northern America', 'Northern Europe',    'Polynesia', 'South America', 'South-eastern Asia', 'Southern Africa',    'Southern Asia', 'Southern Europe', 'Western Africa', 'Western Asia',    'Western Europe',]NAME_ALIASES = {    'United States': 'USA', 'United States of America': 'USA',    'Republic of Korea': 'South Korea', 'Czechia': 'Czech Republic',    'Burma': 'Myanmar', 'Bahamas': 'The Bahamas',    'Cabo Verde': 'Cape Verde', 'Timor-Leste': 'East Timor',    'Russian Federation': 'Russia', 'Syrian Arab Republic': 'Syria',    'Viet Nam': 'Vietnam', 'Türkiye': 'Turkey',    "Côte d'Ivoire": "Cote d'Ivoire",}TAIWAN_NAMES = {'Taiwan', 'Taiwan*', 'Taiwan (Province of China)'}# UNSD 文件只读。最细层级：有 Intermediate Region 时使用它，否则使用 Sub-region。df_un = pd.read_csv('mapping/UNSD — Methodology.csv', sep=';', dtype=str)df_un = df_un[df_un['Country or Area'].notna()].copy()intermediate = df_un['Intermediate Region Name'].fillna('').str.strip()subregion = df_un['Sub-region Name'].fillna('').str.strip()df_un['UN_Region'] = intermediate.where(intermediate.ne(''), subregion)unassigned = df_un.loc[df_un['UN_Region'].eq(''), 'Country or Area'].tolist()if unassigned != ['Antarctica']:    raise ValueError(f'UNSD 中出现非预期无区域对象: {unassigned}')df_un = df_un[df_un['UN_Region'].ne('')].copy()actual_regions = sorted(df_un['UN_Region'].unique())if actual_regions != UN_REGION_ORDER:    raise ValueError('UNSD 的22个最细区域与固定 Region_Index 顺序不一致。')region_continent = df_un[['UN_Region', 'Region Name']].drop_duplicates()if region_continent.groupby('UN_Region')['Region Name'].nunique().ne(1).any():    raise ValueError('一个最细区域对应多个洲级 Region Name。')region_to_continent = region_continent.set_index('UN_Region')['Region Name'].to_dict()df_region = pd.DataFrame({    'Region_Index': range(22),    'Region_Name': UN_REGION_ORDER,    'Type': 'Sub-region',    'UN_Region': UN_REGION_ORDER,    'Continent': [region_to_continent[name] for name in UN_REGION_ORDER],})region_to_idx = dict(zip(df_region['Region_Name'], df_region['Region_Index']))iso2_to_iso3 = (    df_un.dropna(subset=['ISO-alpha2 Code', 'ISO-alpha3 Code'])    .drop_duplicates('ISO-alpha2 Code')    .set_index('ISO-alpha2 Code')['ISO-alpha3 Code']    .to_dict())valid_iso3 = set(df_un['ISO-alpha3 Code'].dropna())# TW/TWN/Taiwan 作为中国数据归并到 CHN，不建立独立实体。iso2_to_iso3['TW'] = 'CHN'df_iso_names = pd.read_csv('mapping/country_iso_mapping.csv')name_to_iso2 = dict(zip(df_iso_names['Country_Name'], df_iso_names['ISO_Alpha2']))df_ihme_codes = pd.read_csv(    'daily_case_data/IHME_population_daily_cases.csv', usecols=['code'])ihme_codes = set(    df_ihme_codes.loc[df_ihme_codes['code'].astype(str).str.len().eq(2), 'code'].dropna())ihme_iso3 = {iso2_to_iso3[code] for code in ihme_codes if code in iso2_to_iso3}ihme_unresolved = sorted(code for code in ihme_codes if code not in iso2_to_iso3)df_sequence_locations = pd.read_csv(    'sequences_data/covid_2019_2020_metadata.csv', usecols=['Location'])sequence_names = set(df_sequence_locations['Location'].str.split(' / ').str[1].dropna())sequence_iso3 = set()sequence_unresolved = []for raw_name in sorted(sequence_names):    if raw_name in TAIWAN_NAMES:        iso3 = 'CHN'    else:        iso2 = name_to_iso2.get(raw_name)        if iso2 is None:            iso2 = name_to_iso2.get(NAME_ALIASES.get(raw_name, raw_name))        iso3 = iso2_to_iso3.get(iso2)    if iso3:        sequence_iso3.add(iso3)    else:        sequence_unresolved.append(raw_name)# -------------------- Flight 国家集合 --------------------flight_header = pd.read_csv(    'flight_data/all_flight_with_passengers_unified.csv', nrows=0).columnsflight_country_chunks = []for chunk in pd.read_csv(    'flight_data/all_flight_with_passengers_unified.csv',    usecols=['origin_country', 'dest_country'],    chunksize=500_000,):    flight_country_chunks.append(        set(chunk['origin_country'].dropna()) | set(chunk['dest_country'].dropna())    )flight_raw_codes = set().union(*flight_country_chunks)flight_normalized_codes = {'CHN' if code == 'TWN' else code for code in flight_raw_codes}flight_iso3 = flight_normalized_codes & valid_iso3flight_unresolved = sorted(    code for code in flight_raw_codes    if ('CHN' if code == 'TWN' else code) not in valid_iso3)# -------------------- GTMD2 国家集合 --------------------if not GTMD_PATH.exists():    raise FileNotFoundError(f'未找到 GTMD2: {GTMD_PATH.resolve()}')gtmd_raw_codes = set()for chunk in pd.read_csv(    GTMD_PATH,    usecols=['year', 'iso3code_i', 'iso3code_j'],    dtype={'year': 'int16', 'iso3code_i': 'string', 'iso3code_j': 'string'},    chunksize=200_000,):    chunk = chunk[chunk['year'].isin(GTMD_YEARS)]    gtmd_raw_codes.update(chunk['iso3code_i'].dropna().astype(str).unique())    gtmd_raw_codes.update(chunk['iso3code_j'].dropna().astype(str).unique())gtmd_normalized_codes = {'CHN' if code == 'TWN' else code for code in gtmd_raw_codes}gtmd_iso3 = gtmd_normalized_codes & valid_iso3gtmd_unresolved = sorted(    code for code in gtmd_raw_codes    if ('CHN' if code == 'TWN' else code) not in valid_iso3)# 统一交集：Flight + GTMD2 + IHME + Sequences 四者交集unified_intersection_iso3 = ihme_iso3 & flight_iso3 & gtmd_iso3 & sequence_iso3def save_scenario_mapping(output_dir, selected_iso3, mobility_name, mobility_set, mobility_unresolved):    selected = df_un[df_un['ISO-alpha3 Code'].isin(selected_iso3)].copy()    selected['UN_Level'] = (        selected['Intermediate Region Name'].fillna('').str.strip().ne('').map({            True: 'Intermediate Region', False: 'Sub-region',        })    )    mapping = pd.DataFrame({        'Country': selected['Country or Area'],        'M49_Code': selected['M49 Code'],        'ISO_Alpha2': selected['ISO-alpha2 Code'],        'ISO_Alpha3': selected['ISO-alpha3 Code'],        'UN_Region': selected['UN_Region'],        'UN_Level': selected['UN_Level'],        'UN_Region_Code': selected['Region Code'],        'UN_Subregion_Code': selected['Sub-region Code'],        'UN_Intermediate_Region_Code': selected['Intermediate Region Code'],    })    mapping['Continent'] = mapping['UN_Region'].map(region_to_continent)    mapping['Region_Index'] = mapping['UN_Region'].map(region_to_idx).astype(int)    mapping['Region_Name'] = mapping['UN_Region']    mapping['Is_Independent'] = False    mapping = mapping.sort_values(['Region_Index', 'Country'])    audit = pd.DataFrame([        {'Source': 'IHME', 'Recognized_Countries': len(ihme_iso3),         'Unresolved_Identifiers': '|'.join(ihme_unresolved),         'Intersection_Countries': len(selected_iso3)},        {'Source': mobility_name, 'Recognized_Countries': len(mobility_set),         'Unresolved_Identifiers': '|'.join(mobility_unresolved),         'Intersection_Countries': len(selected_iso3)},        {'Source': 'Sequences', 'Recognized_Countries': len(sequence_iso3),         'Unresolved_Identifiers': '|'.join(sequence_unresolved),         'Intersection_Countries': len(selected_iso3)},    ])    df_region.to_csv(output_dir / 'region_mapping.csv', index=False)    mapping.to_csv(output_dir / 'country_to_region_mapping.csv', index=False)    # audit.to_csv(output_dir / 'country_intersection_audit.csv', index=False)    # pd.DataFrame([    #     {'Raw_Identifier': 'TW/TWN/Taiwan', 'Normalized_To': 'China/CHN',    #      'UN_Region': 'Eastern Asia'},    # ]).to_csv(output_dir / 'entity_override_audit.csv', index=False)    print(f'\n{mobility_name} 版本: {len(mapping)} 个交集国家，22个区域节点')    print(audit.to_string(index=False))    return mapping# 统一输出目录UNIFIED_OUTPUT_DIR = FLIGHT_OUTPUT_DIR  # 共用 Flight 目录unified_country_mapping = save_scenario_mapping(    UNIFIED_OUTPUT_DIR, unified_intersection_iso3,    'Unified', unified_intersection_iso3, ['']*len(unified_intersection_iso3))del df_ihme_codes, df_sequence_locations, flight_country_chunks


Flight 版本: 157 个交集国家，22个区域节点
   Source  Recognized_Countries Unresolved_Identifiers  Intersection_Countries
     IHME                   165                                            157
   Flight                   232                    XKX                     157
Sequences                   186         Kosovo|unknown                     157

GTMD2 版本: 159 个交集国家，22个区域节点
   Source  Recognized_Countries Unresolved_Identifiers  Intersection_Countries
     IHME                   165                                            159
    GTMD2                   238                    ATA                     159
Sequences                   186         Kosovo|unknown                     159


### 2. 分别绘制两套 UN 区域地图（可选）


In [3]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


shape_candidates = [
    Path('mapping/map_cache/ne_110m_admin_0_countries.shp'),
    Path('map_cache/ne_110m_admin_0_countries.shp'),
]
shape_path = next((path for path in shape_candidates if path.exists()), None)
if shape_path is None:
    print('跳过地图：未找到本地 Natural Earth shapefile。')
else:
    world_base = gpd.read_file(shape_path)
    if 'ADM0_A3' not in world_base.columns:
        raise ValueError('Natural Earth shapefile 缺少 ADM0_A3 字段。')

    for scenario, output_dir in {
        'flight': FLIGHT_OUTPUT_DIR,
        'gtmd': GTMD_OUTPUT_DIR,
    }.items():
        mapping = pd.read_csv(output_dir / 'country_to_region_mapping.csv')
        iso3_to_region_map = dict(zip(mapping['ISO_Alpha3'], mapping['Region_Name']))
        world = world_base.copy()
        world['Region_Name'] = world['ADM0_A3'].map(iso3_to_region_map)
        palette = plt.get_cmap('tab20')
        colors = {name: palette(index % 20) for index, name in enumerate(UN_REGION_ORDER)}
        world['color'] = world['Region_Name'].map(colors).fillna('#E0E0E0')
        fig, ax = plt.subplots(figsize=(24, 14))
        world.plot(ax=ax, color=world['color'], edgecolor='white', linewidth=0.25)
        legend = [Patch(facecolor=colors[name], label=name) for name in UN_REGION_ORDER]
        ax.legend(handles=legend, loc='lower left', fontsize=7, ncol=3)
        ax.set_title(f'UN Regions — {scenario.upper()} country intersection')
        ax.axis('off')
        fig.tight_layout()
        fig.savefig(output_dir / 'un_region_map.png', dpi=250, bbox_inches='tight')
        plt.close(fig)


### 3. 比较两套国家交集的区域组成


In [4]:
comparison_rows = []
for scenario, output_dir in {'flight': FLIGHT_OUTPUT_DIR, 'gtmd': GTMD_OUTPUT_DIR}.items():
    mapping = pd.read_csv(output_dir / 'country_to_region_mapping.csv')
    counts = mapping.groupby(['Region_Index', 'Region_Name']).size()
    for region_index, region_name in enumerate(UN_REGION_ORDER):
        comparison_rows.append({
            'Scenario': scenario,
            'Region_Index': region_index,
            'Region_Name': region_name,
            'Country_Count': int(counts.get((region_index, region_name), 0)),
        })
df_scenario_region_counts = pd.DataFrame(comparison_rows)
print(df_scenario_region_counts.pivot(
    index=['Region_Index', 'Region_Name'], columns='Scenario', values='Country_Count'
).fillna(0).astype(int).to_string())


Scenario                                flight  gtmd
Region_Index Region_Name                            
0            Australia and New Zealand       2     2
1            Caribbean                       9     9
2            Central America                 6     6
3            Central Asia                    3     3
4            Eastern Africa                 11    11
5            Eastern Asia                    5     5
6            Eastern Europe                 10    10
7            Melanesia                       2     2
8            Micronesia                      1     1
9            Middle Africa                   7     7
10           Northern Africa                 6     6
11           Northern America                2     2
12           Northern Europe                10    10
13           Polynesia                       0     0
14           South America                  12    12
15           South-eastern Asia              9     9
16           Southern Africa                 4

### 4. 验证两套固定区域映射


In [5]:
for scenario, output_dir in {'flight': FLIGHT_OUTPUT_DIR, 'gtmd': GTMD_OUTPUT_DIR}.items():
    mapping = pd.read_csv(output_dir / 'region_mapping.csv').sort_values('Region_Index')
    assert len(mapping) == 22
    assert mapping['Region_Index'].tolist() == list(range(22))
    assert mapping['Region_Name'].tolist() == UN_REGION_ORDER
    assert mapping.loc[mapping['Region_Name'].eq('Eastern Asia'), 'Region_Index'].item() == 5
    print(f'{scenario}: 固定22区域映射验证通过')


flight: 固定22区域映射验证通过
gtmd: 固定22区域映射验证通过


In [6]:
set_comparison = pd.DataFrame({
    'Metric': ['Countries', 'Flight-only', 'GTMD-only', 'Shared'],
    'Value': [
        len(flight_intersection_iso3 | gtmd_intersection_iso3),
        len(flight_intersection_iso3 - gtmd_intersection_iso3),
        len(gtmd_intersection_iso3 - flight_intersection_iso3),
        len(flight_intersection_iso3 & gtmd_intersection_iso3),
    ],
})
print(set_comparison.to_string(index=False))


     Metric  Value
  Countries    159
Flight-only      0
  GTMD-only      2
     Shared    157


### 5A. 统一版本：病例、人口、序列与航班流

In [7]:
import numpy as np


CUTOFF_DATE = pd.Timestamp('2020-06-01')


def map_sequence_name_to_region(name, name_to_iso2_map, iso2_to_region_map):
    if name in TAIWAN_NAMES:
        return iso2_to_region_map.get('TW')
    iso2 = name_to_iso2_map.get(name)
    if iso2 is None:
        iso2 = name_to_iso2_map.get(NAME_ALIASES.get(name, name))
    return iso2_to_region_map.get(iso2)


def aggregate_health_and_sequences(country_mapping_path, output_dir):
    processed_dir = output_dir / 'processed_data'
    mapping = pd.read_csv(country_mapping_path)
    regions = pd.read_csv(output_dir / 'region_mapping.csv').sort_values('Region_Index')
    region_names = regions['Region_Name'].tolist()
    region_index = dict(zip(regions['Region_Name'], regions['Region_Index']))
    iso2_region = dict(zip(mapping['ISO_Alpha2'], mapping['Region_Name']))
    iso3_region = dict(zip(mapping['ISO_Alpha3'], mapping['Region_Name']))
    if iso3_region.get('CHN') != 'Eastern Asia':
        raise ValueError('中国必须映射到 Eastern Asia。')
    iso2_region['TW'] = 'Eastern Asia'
    iso3_region['TWN'] = 'Eastern Asia'

    audit = []
    ihme = pd.read_csv('daily_case_data/IHME_population_daily_cases.csv')
    ihme['date'] = pd.to_datetime(ihme['date'])
    ihme = ihme[ihme['code'].astype(str).str.len().eq(2)].copy()
    ihme['region'] = ihme['code'].map(iso2_region)
    audit.append({
        'Source': 'IHME', 'Total_Records': len(ihme),
        'Mapped_Records': int(ihme['region'].notna().sum()),
        'Unmapped_Records': int(ihme['region'].isna().sum()),
    })
    ihme = ihme[ihme['region'].notna() & ihme['date'].le(CUTOFF_DATE)].copy()

    daily = ihme.pivot_table(
        values='daily_cases', index='date', columns='region', aggfunc='sum', fill_value=0
    ).reindex(columns=region_names, fill_value=0)
    daily.index.name = 'Date'
    daily.to_csv(processed_dir / 'covid19_daily_new_by_region.csv')

    latest_population = (
        ihme.sort_values('date').dropna(subset=['population']).groupby('code', as_index=False).tail(1)
    )
    population = (
        latest_population.groupby('region', as_index=False)['population'].sum()
        .rename(columns={'region': 'Region', 'population': 'Population'})
        .set_index('Region').reindex(region_names, fill_value=0).reset_index()
    )
    population['Region_Index'] = population['Region'].map(region_index).astype(int)
    population[['Region_Index', 'Region', 'Population']].sort_values('Region_Index').to_csv(
        processed_dir / 'region_population_idx.csv', index=False
    )
    population[['Region', 'Population']].to_csv(
        processed_dir / 'population_by_region.csv', index=False
    )

    iso_names = pd.read_csv('mapping/country_iso_mapping.csv')
    name_iso2 = dict(zip(iso_names['Country_Name'], iso_names['ISO_Alpha2']))
    sequences = pd.read_csv(
        'sequences_data/covid_2019_2020_metadata.csv',
        usecols=['Collection date', 'Location'],
    )
    sequences['country_raw'] = sequences['Location'].str.split(' / ').str[1]
    sequences['date'] = pd.to_datetime(sequences['Collection date'], errors='coerce')
    sequences['region'] = sequences['country_raw'].map(
        lambda name: map_sequence_name_to_region(name, name_iso2, iso2_region)
    )
    audit.append({
        'Source': 'Sequences', 'Total_Records': len(sequences),
        'Mapped_Records': int(sequences['region'].notna().sum()),
        'Unmapped_Records': int(sequences['region'].isna().sum()),
    })
    sequences = sequences[
        sequences['region'].notna() & sequences['date'].notna()
        & sequences['date'].le(CUTOFF_DATE)
    ].copy()
    date_range = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
    sequence_daily = (
        sequences.groupby(['date', 'region']).size().unstack(fill_value=0)
        .reindex(index=date_range, columns=region_names, fill_value=0).astype(int)
    )
    sequence_daily.index.name = 'Date'
    sequence_daily.to_csv(processed_dir / 'covid19_sequences_daily_by_region.csv')

    cases_7d = daily.rolling(7, min_periods=1).sum()
    sequences_7d = sequence_daily.rolling(7, min_periods=1).sum()
    sampling_7d = (sequences_7d / cases_7d.where(cases_7d.ne(0)) * 100).clip(upper=100)
    sampling_7d.to_csv(processed_dir / 'region_sampling_rate_7d.csv', float_format='%.10f')
    static_sampling = (sequence_daily.sum() / daily.sum().replace(0, np.nan) * 100).clip(upper=100).fillna(0)
    pd.DataFrame({'Region': region_names, 'Sampling_Rate_%': static_sampling.reindex(region_names).values}).to_csv(
        processed_dir / 'region_sampling_rate.csv', index=False
    )
    return region_names, region_index, iso2_region, iso3_region, audit


flight_region_names, flight_region_to_idx, _, flight_iso3_to_region, flight_audit = (
    aggregate_health_and_sequences(
        FLIGHT_OUTPUT_DIR / 'country_to_region_mapping.csv', FLIGHT_OUTPUT_DIR
    )
)

flight_path = Path('flight_data/all_flight_with_passengers_unified.csv')
flight_columns = pd.read_csv(flight_path, nrows=0).columns
passenger_column = 'passengers_rounded' if 'passengers_rounded' in flight_columns else 'passengers'
flight_parts = []
flight_total_rows = 0
flight_mapped_rows = 0
for chunk in pd.read_csv(
    flight_path,
    usecols=['year', 'month', 'origin_country', 'dest_country', passenger_column],
    chunksize=500_000,
):
    chunk = chunk[chunk['year'].isin([2019, 2020])].copy()
    if chunk.empty:
        continue
    flight_total_rows += len(chunk)
    chunk['origin_iso3'] = chunk['origin_country'].replace({'TWN': 'CHN'})
    chunk['dest_iso3'] = chunk['dest_country'].replace({'TWN': 'CHN'})
    chunk['origin'] = chunk['origin_iso3'].map(flight_iso3_to_region)
    chunk['destination'] = chunk['dest_iso3'].map(flight_iso3_to_region)
    mapped = chunk['origin'].notna() & chunk['destination'].notna()
    chunk = chunk[mapped]
    flight_mapped_rows += len(chunk)
    flight_parts.append(
        chunk.groupby(['year', 'month', 'origin', 'destination'])[passenger_column]
        .sum().reset_index()
    )

flight_by_region = (
    pd.concat(flight_parts, ignore_index=True)
    .groupby(['year', 'month', 'origin', 'destination'])[passenger_column]
    .sum().reset_index().rename(columns={passenger_column: 'passengers'})
)
flight_by_region.to_csv(
    FLIGHT_OUTPUT_DIR / 'processed_data' / 'flight_by_region.csv', index=False
)
flight_idx = flight_by_region.assign(
    ORIGIN_REGION=flight_by_region['origin'].map(flight_region_to_idx).astype(int),
    DEST_REGION=flight_by_region['destination'].map(flight_region_to_idx).astype(int),
).rename(columns={'year': 'YEAR', 'month': 'MONTH', 'passengers': 'PASSENGERS'})
flight_idx[['YEAR', 'MONTH', 'ORIGIN_REGION', 'DEST_REGION', 'PASSENGERS']].to_csv(
    FLIGHT_OUTPUT_DIR / 'processed_data' / 'region_flow_monthly_idx.csv', index=False
)
flight_audit.append({
    'Source': 'Flight_OD', 'Total_Records': flight_total_rows,
    'Mapped_Records': flight_mapped_rows,
    'Unmapped_Records': flight_total_rows - flight_mapped_rows,
})
# pd.DataFrame(flight_audit).to_csv(
#     FLIGHT_OUTPUT_DIR / 'data_source_mapping_audit.csv', index=False
# )
print(f'Flight 区域月流记录: {len(flight_idx)}')


Flight 区域月流记录: 9219


### 5B. GTMD2 年度聚合与月度分配（使用统一交集）

In [8]:
gtmd_region_names, gtmd_region_to_idx, _, gtmd_iso3_to_region, gtmd_audit = (    aggregate_health_and_sequences(        GTMD_OUTPUT_DIR / 'country_to_region_mapping.csv', GTMD_OUTPUT_DIR    ))GTMD_FLOW_COLUMNS = ['gtmd2_trips_s1', 'gtmd2_trips_s2', 'gtmd2_vflow_int']GTMD_USECOLS = ['year', 'iso3code_i', 'iso3code_j', *GTMD_FLOW_COLUMNS]gtmd_parts = []gtmd_total_rows = 0gtmd_mapped_rows = 0gtmd_nonmissing = {column: 0 for column in GTMD_FLOW_COLUMNS}for chunk in pd.read_csv(    GTMD_PATH, usecols=GTMD_USECOLS,    dtype={'year': 'int16', 'iso3code_i': 'string', 'iso3code_j': 'string'},    chunksize=200_000,):    chunk = chunk[chunk['year'].isin(GTMD_YEARS)].copy()    if chunk.empty:        continue    gtmd_total_rows += len(chunk)    chunk['origin_iso3'] = chunk['iso3code_i'].replace({'TWN': 'CHN'})    chunk['dest_iso3'] = chunk['iso3code_j'].replace({'TWN': 'CHN'})    chunk['origin'] = chunk['origin_iso3'].map(gtmd_iso3_to_region)    chunk['destination'] = chunk['dest_iso3'].map(gtmd_iso3_to_region)    mapped = chunk['origin'].notna() & chunk['destination'].notna()    chunk = chunk[mapped]    gtmd_mapped_rows += len(chunk)    for column in GTMD_FLOW_COLUMNS:        gtmd_nonmissing[column] += int(chunk[column].notna().sum())    gtmd_parts.append(        chunk.groupby(['year', 'origin', 'destination'])[GTMD_FLOW_COLUMNS]        .sum(min_count=1).reset_index()    )gtmd_annual = (    pd.concat(gtmd_parts, ignore_index=True)    .groupby(['year', 'origin', 'destination'])[GTMD_FLOW_COLUMNS]    .sum(min_count=1).reset_index())gtmd_annual['ORIGIN_REGION'] = gtmd_annual['origin'].map(gtmd_region_to_idx).astype(int)gtmd_annual['DEST_REGION'] = gtmd_annual['destination'].map(gtmd_region_to_idx).astype(int)gtmd_annual = gtmd_annual.rename(columns={    'year': 'YEAR', 'origin': 'ORIGIN_REGION_NAME',    'destination': 'DEST_REGION_NAME',})[    ['YEAR', 'ORIGIN_REGION', 'DEST_REGION', 'ORIGIN_REGION_NAME',     'DEST_REGION_NAME', *GTMD_FLOW_COLUMNS]].sort_values(['YEAR', 'ORIGIN_REGION', 'DEST_REGION'])gtmd_annual.to_csv(    GTMD_OUTPUT_DIR / 'processed_data' / 'gtmd2_region_annual_2019_2020.csv', index=False)# GTMD2 是年度数据；月度拆分只借用 Flight 版本的区域季节性，不改变 GTMD2 年总量。seasonality = pd.read_csv(    FLIGHT_OUTPUT_DIR / 'processed_data' / 'region_flow_monthly_idx.csv')od_month = seasonality.groupby(    ['YEAR', 'MONTH', 'ORIGIN_REGION', 'DEST_REGION'], as_index=False)['PASSENGERS'].sum()od_total = od_month.groupby(    ['YEAR', 'ORIGIN_REGION', 'DEST_REGION'], as_index=False)['PASSENGERS'].sum().rename(columns={'PASSENGERS': 'OD_TOTAL'})origin_month = od_month.groupby(    ['YEAR', 'MONTH', 'ORIGIN_REGION'], as_index=False)['PASSENGERS'].sum().rename(columns={'PASSENGERS': 'ORIGIN_MONTH'})origin_total = origin_month.groupby(    ['YEAR', 'ORIGIN_REGION'], as_index=False)['ORIGIN_MONTH'].sum().rename(columns={'ORIGIN_MONTH': 'ORIGIN_TOTAL'})global_month = od_month.groupby(['YEAR', 'MONTH'], as_index=False)['PASSENGERS'].sum().rename(    columns={'PASSENGERS': 'GLOBAL_MONTH'})global_total = global_month.groupby('YEAR', as_index=False)['GLOBAL_MONTH'].sum().rename(    columns={'GLOBAL_MONTH': 'GLOBAL_TOTAL'})gtmd_monthly = gtmd_annual.merge(pd.DataFrame({'MONTH': range(1, 13)}), how='cross')gtmd_monthly = gtmd_monthly.merge(    od_month.rename(columns={'PASSENGERS': 'OD_MONTH'}),    on=['YEAR', 'MONTH', 'ORIGIN_REGION', 'DEST_REGION'], how='left',).merge(od_total, on=['YEAR', 'ORIGIN_REGION', 'DEST_REGION'], how='left')gtmd_monthly = gtmd_monthly.merge(    origin_month, on=['YEAR', 'MONTH', 'ORIGIN_REGION'], how='left').merge(origin_total, on=['YEAR', 'ORIGIN_REGION'], how='left')gtmd_monthly = gtmd_monthly.merge(    global_month, on=['YEAR', 'MONTH'], how='left').merge(global_total, on='YEAR', how='left')od_ok = gtmd_monthly['OD_TOTAL'].fillna(0).gt(0)origin_ok = gtmd_monthly['ORIGIN_TOTAL'].fillna(0).gt(0)global_ok = gtmd_monthly['GLOBAL_TOTAL'].fillna(0).gt(0)od_share = gtmd_monthly['OD_MONTH'].fillna(0) / gtmd_monthly['OD_TOTAL'].where(od_ok)origin_share = gtmd_monthly['ORIGIN_MONTH'].fillna(0) / gtmd_monthly['ORIGIN_TOTAL'].where(origin_ok)global_share = gtmd_monthly['GLOBAL_MONTH'].fillna(0) / gtmd_monthly['GLOBAL_TOTAL'].where(global_ok)    [od_ok, origin_ok, global_ok], ['od_flight_share', 'origin_flight_share', 'global_flight_share'], default='equal_1_12'
    [od_ok, origin_ok, global_ok], ['OD', 'ORIGIN', 'GLOBAL'], default='EQUAL_1_12')gtmd_monthly['MONTHLY_SHARE'] = np.select(    [od_ok, origin_ok, global_ok], [od_share, origin_share, global_share], default=1 / 12)for column in GTMD_FLOW_COLUMNS:    gtmd_monthly[column] = gtmd_monthly[column] * gtmd_monthly['MONTHLY_SHARE']gtmd_monthly_output = gtmd_monthly[    ['YEAR', 'MONTH', 'ORIGIN_REGION', 'DEST_REGION', 'ORIGIN_REGION_NAME',     'DEST_REGION_NAME', 'ALLOCATION_METHOD', 'MONTHLY_SHARE', *GTMD_FLOW_COLUMNS]].sort_values(['YEAR', 'MONTH', 'ORIGIN_REGION', 'DEST_REGION'])gtmd_monthly_output.to_csv(    GTMD_OUTPUT_DIR / 'processed_data' / 'gtmd2_region_monthly_2019_2020.csv', index=False)metric_audit = pd.DataFrame([    {        'Metric': column, 'Selected_Rows': gtmd_mapped_rows,        'Nonmissing_Rows': gtmd_nonmissing[column],        'Coverage_Percent': 100 * gtmd_nonmissing[column] / gtmd_mapped_rows,        'Regional_Annual_Rows': int(gtmd_annual[column].notna().sum()),        'Regional_Total': gtmd_annual[column].sum(min_count=1),    }    for column in GTMD_FLOW_COLUMNS])metric_audit.to_csv(GTMD_OUTPUT_DIR / 'gtmd2_region_aggregation_audit.csv', index=False)gtmd_audit.append({    'Source': 'GTMD2_OD', 'Total_Records': gtmd_total_rows,    'Mapped_Records': gtmd_mapped_rows,    'Unmapped_Records': gtmd_total_rows - gtmd_mapped_rows,})pd.DataFrame(gtmd_audit).to_csv(    GTMD_OUTPUT_DIR / 'data_source_mapping_audit.csv', index=False)print(f'GTMD2 年度区域流: {len(gtmd_annual)} 条')print(f'GTMD2 月度区域流: {len(gtmd_monthly_output)} 条')print(metric_audit.to_string(index=False))

GTMD2 年度区域流: 880 条
GTMD2 月度区域流: 10560 条
         Metric  Selected_Rows  Nonmissing_Rows  Coverage_Percent  Regional_Annual_Rows  Regional_Total
 gtmd2_trips_s1          50880            49338         96.969340                   880    1.425924e+10
 gtmd2_trips_s2          50880            49338         96.969340                   880    1.425924e+10
gtmd2_vflow_int          50880            24215         47.592374                   809    4.573503e+09


In [9]:
import matplotlib.pyplot as plt


for scenario, output_dir in {'flight': FLIGHT_OUTPUT_DIR, 'gtmd': GTMD_OUTPUT_DIR}.items():
    figures_dir = output_dir / 'figures'
    processed_dir = output_dir / 'processed_data'
    cases = pd.read_csv(
        processed_dir / 'covid19_daily_new_by_region.csv', index_col='Date', parse_dates=True
    )
    sequences = pd.read_csv(
        processed_dir / 'covid19_sequences_daily_by_region.csv', index_col='Date', parse_dates=True
    )
    fig, axes = plt.subplots(2, 1, figsize=(18, 12), sharex=True)
    cases.rolling(7, min_periods=1).mean().plot(ax=axes[0], legend=False)
    sequences.rolling(7, min_periods=1).mean().plot(ax=axes[1], legend=False)
    axes[0].set_title(f'{scenario.upper()} intersection — cases (7-day MA)')
    axes[1].set_title(f'{scenario.upper()} intersection — sequences (7-day MA)')
    fig.tight_layout()
    fig.savefig(figures_dir / 'fig_cases_sequences_comparison.png', dpi=180)
    plt.close(fig)

flight_flow = pd.read_csv(FLIGHT_OUTPUT_DIR / 'processed_data' / 'region_flow_monthly_idx.csv')
gtmd_flow = pd.read_csv(GTMD_OUTPUT_DIR / 'processed_data' / 'gtmd2_region_annual_2019_2020.csv')
print('Flight 与 GTMD2 图形已分别保存。')


Flight 与 GTMD2 图形已分别保存。


In [10]:
for scenario, output_dir in {'flight': FLIGHT_OUTPUT_DIR, 'gtmd': GTMD_OUTPUT_DIR}.items():
    regions = pd.read_csv(output_dir / 'region_mapping.csv').sort_values('Region_Index')
    population = pd.read_csv(output_dir / 'processed_data' / 'region_population_idx.csv')
    cases = pd.read_csv(output_dir / 'processed_data' / 'covid19_daily_new_by_region.csv', index_col='Date')
    sequences = pd.read_csv(output_dir / 'processed_data' / 'covid19_sequences_daily_by_region.csv', index_col='Date')
    expected = regions['Region_Name'].tolist()
    assert len(regions) == 22 and regions['Region_Index'].tolist() == list(range(22))
    assert len(population) == 22
    assert cases.columns.tolist() == expected
    assert sequences.columns.tolist() == expected
    print(f'{scenario}: 健康数据固定22区域验证通过')

flight_flow = pd.read_csv(FLIGHT_OUTPUT_DIR / 'processed_data' / 'region_flow_monthly_idx.csv')
assert flight_flow.columns.tolist() == ['YEAR', 'MONTH', 'ORIGIN_REGION', 'DEST_REGION', 'PASSENGERS']

gtmd_annual_check = pd.read_csv(GTMD_OUTPUT_DIR / 'processed_data' / 'gtmd2_region_annual_2019_2020.csv')
gtmd_monthly_check = pd.read_csv(GTMD_OUTPUT_DIR / 'processed_data' / 'gtmd2_region_monthly_2019_2020.csv')
for metric in ['gtmd2_trips_s1', 'gtmd2_trips_s2', 'gtmd2_vflow_int']:
    annual = gtmd_annual_check.set_index(['YEAR', 'ORIGIN_REGION', 'DEST_REGION'])[metric].sort_index()
    monthly = gtmd_monthly_check.groupby(['YEAR', 'ORIGIN_REGION', 'DEST_REGION'])[metric].sum(min_count=1).sort_index()
    assert np.allclose(annual.fillna(0), monthly.fillna(0), rtol=1e-10, atol=1e-6)
print('两套流量输入及 GTMD2 月度守恒验证通过')


flight: 健康数据固定22区域验证通过
gtmd: 健康数据固定22区域验证通过
两套流量输入及 GTMD2 月度守恒验证通过
